In [1]:
import pandas as pd
from top2vec import Top2Vec
import os
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from umap.umap_ import UMAP
import plotly.express as px

/opt/anaconda3/envs/kkk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
stopwords=stopwords.words('english')
lemmatizer = WordNetLemmatizer()

In [ ]:
data=pd.read_feather('/Volumes/T7/chroniclingamerica/american-stories/kkk-revival-stanza-lynch.feather')

In [ ]:
data.shape

In [ ]:
data.head(2)

In [ ]:
data['article_stop'] = data['article'].str.lower().str.split().apply(lambda x: [word for word in x if word not in stopwords])


In [ ]:
data['article_lemma'] = data['article_stop'].apply(lambda x: [WordNetLemmatizer().lemmatize(word) for word in x])

In [ ]:
data['article_lemma_string']=data['article_lemma'].apply(lambda x: ' '.join(x))

In [ ]:
Counter(data['article_lemma'].explode()).most_common(50)

In [ ]:
model=Top2Vec(documents=data['article_lemma_string'].tolist(), speed="learn", workers=8)

In [ ]:
topic_sizes, topic_nums = model.get_topic_sizes()

In [ ]:
print(len(topic_sizes), len(topic_nums))

In [ ]:
id_dic={}
topic_id={}
for element in zip(topic_nums, topic_sizes):
    documents, document_scores, document_ids = model.search_documents_by_topic(topic_num=element[0], num_docs=element[1])
    for score, id in zip(document_scores, document_ids):
        id_dic[id]=score
        topic_id[id]=element[0]

In [ ]:
topic_words, word_scores, topic_scores = model.get_topics(len(topic_sizes))

In [ ]:
df_words=pd.DataFrame(topic_words).transpose()
df_words.columns=topic_nums
df_words.columns = df_words.columns.astype(str)

In [ ]:
df_words[['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']].iloc[:20]

In [ ]:
data['topic_id']=data.index.map(topic_id) #map topic id to each document
data['topic_score']=data.index.map(id_dic) #map topic score to each document

In [ ]:
Counter(data['topic_id']).most_common(10)

In [ ]:
data[['article', 'topic_id', 'topic_score']].head(10)

In [ ]:
data['vector']=model.document_vectors.tolist()

In [ ]:
data.to_feather('/Volumes/T7/chroniclingamerica/american-stories/kkk-revival-top2vec.feather')

In [2]:
data=pd.read_feather('/Volumes/T7/chroniclingamerica/american-stories/kkk-revival-top2vec.feather')

In [4]:
embedding = UMAP().fit_transform(data['vector'].tolist()) #reduce dimensionality of vector representation

In [5]:
clusterable_embedding = UMAP(
    n_neighbors=30,
    min_dist=0.0,
    n_components=2,
    random_state=42,
).fit_transform(embedding.data)

/opt/anaconda3/envs/kkk/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [7]:
fig = px.scatter(
    data, 
    x=clusterable_embedding[:, 0], 
    y=clusterable_embedding[:, 1], 
    color='topic_id',  # Specify the column for color
    hover_data=['topic_id']
)

fig.update_layout(
    autosize=False,
    width=1000,
    height=1000,
    title="UMAP projection of the document vectors"
)

# fig.show()
fig.write_html("/Volumes/T7/chroniclingamerica/american-stories/interactive-plot.html")
#when hover over the dots, the topic id and doi will show up as well as x and y coordinates